# CEFR Text Generation using GAN - Improved Version
High-performance GAN model for generating CEFR-leveled text datasets with enhanced accuracy.

In [17]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import re
import warnings
warnings.filterwarnings('ignore')

print("TensorFlow version:", tf.__version__)

TensorFlow version: 2.16.2


## 1. Data Loading and Preprocessing Functions

In [18]:
def LoadCefrData(texts_path, vocab_path, grammar_path):
    texts_df = pd.read_csv(texts_path)
    print(f"Loaded {len(texts_df)} texts")
    print(f"CEFR Levels: {texts_df['label'].unique()}")
    
    vocab_df = pd.read_csv(vocab_path)
    print(f"\nLoaded {len(vocab_df)} vocabulary entries")
    
    grammar_df = pd.read_csv(grammar_path)
    print(f"Loaded {len(grammar_df)} grammar patterns")
    
    return texts_df, vocab_df, grammar_df


def CreateVocabularyDict(vocab_df):
    vocab_dict = {}
    
    for level in ['A1', 'A2', 'B1', 'B2', 'C1', 'C2']:
        level_order = {'A1': 1, 'A2': 2, 'B1': 3, 'B2': 4, 'C1': 5, 'C2': 6}
        allowed_levels = [l for l, v in level_order.items() if v <= level_order[level]]
        
        vocab_dict[level] = set(
            vocab_df[vocab_df['CEFR'].isin(allowed_levels)]['headword'].str.lower().unique()
        )
        print(f"{level}: {len(vocab_dict[level])} words")
    
    return vocab_dict


def PreprocessText(text):
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s.,!?\'\-]', ' ', text)
    text = ' '.join(text.split())
    return text


def PrepareSequences(texts_df, max_sequence_length=100, max_words=10000):
    texts_df['processed_text'] = texts_df['text'].apply(PreprocessText)
    
    tokenizer = Tokenizer(num_words=max_words, oov_token='<OOV>')
    tokenizer.fit_on_texts(texts_df['processed_text'])
    
    sequences = tokenizer.texts_to_sequences(texts_df['processed_text'])
    X = pad_sequences(sequences, maxlen=max_sequence_length, padding='post', truncating='post')
    
    label_encoder = {label: idx for idx, label in enumerate(sorted(texts_df['label'].unique()))}
    y = texts_df['label'].map(label_encoder).values
    
    print(f"\nSequence shape: {X.shape}")
    print(f"Labels shape: {y.shape}")
    print(f"Vocabulary size: {len(tokenizer.word_index)}")
    print(f"Label encoding: {label_encoder}")
    
    return X, y, tokenizer, label_encoder

## 2. Generator Function - Enhanced Architecture

In [19]:
def Generator(latent_dim=100, vocab_size=10000, sequence_length=100, num_classes=6, embedding_dim=256):
    noise_input = layers.Input(shape=(latent_dim,), name='noise_input')
    label_input = layers.Input(shape=(1,), name='label_input')
    
    label_embedding = layers.Embedding(num_classes, latent_dim)(label_input)
    label_embedding = layers.Flatten()(label_embedding)
    
    combined_input = layers.Concatenate()([noise_input, label_embedding])
    
    x = layers.Dense(512, activation=layers.LeakyReLU(0.2))(combined_input)
    x = layers.BatchNormalization(momentum=0.8)(x)
    x = layers.Dropout(0.25)(x)
    
    x = layers.Dense(1024, activation=layers.LeakyReLU(0.2))(x)
    x = layers.BatchNormalization(momentum=0.8)(x)
    x = layers.Dropout(0.25)(x)
    
    x = layers.Dense(2048, activation=layers.LeakyReLU(0.2))(x)
    x = layers.BatchNormalization(momentum=0.8)(x)
    x = layers.Dropout(0.25)(x)
    
    x = layers.Dense(sequence_length * embedding_dim, activation=layers.LeakyReLU(0.2))(x)
    x = layers.Reshape((sequence_length, embedding_dim))(x)
    
    x = layers.LSTM(512, return_sequences=True, recurrent_dropout=0.2)(x)
    x = layers.BatchNormalization(momentum=0.8)(x)
    
    x = layers.LSTM(512, return_sequences=True, recurrent_dropout=0.2)(x)
    x = layers.BatchNormalization(momentum=0.8)(x)
    
    x = layers.LSTM(256, return_sequences=True, recurrent_dropout=0.2)(x)
    
    output = layers.TimeDistributed(layers.Dense(vocab_size, activation='softmax'))(x)
    
    model = Model(inputs=[noise_input, label_input], outputs=output, name='Generator')
    return model

## 3. Discriminator Function - Enhanced Architecture

In [20]:
def Discriminator(vocab_size=10000, sequence_length=100, num_classes=6, embedding_dim=256):
    sequence_input = layers.Input(shape=(sequence_length, vocab_size), name='sequence_input')
    label_input = layers.Input(shape=(1,), name='label_input')
    
    label_embedding = layers.Embedding(num_classes, sequence_length)(label_input)
    label_embedding = layers.Flatten()(label_embedding)
    label_embedding = layers.Reshape((sequence_length, 1))(label_embedding)
    
    x = layers.TimeDistributed(layers.Dense(embedding_dim))(sequence_input)
    x = layers.Concatenate()([x, label_embedding])
    
    x = layers.Bidirectional(layers.LSTM(512, return_sequences=True, recurrent_dropout=0.2))(x)
    x = layers.Dropout(0.3)(x)
    
    x = layers.Bidirectional(layers.LSTM(256, return_sequences=True, recurrent_dropout=0.2))(x)
    x = layers.Dropout(0.3)(x)
    
    x = layers.Bidirectional(layers.LSTM(128, recurrent_dropout=0.2))(x)
    
    x = layers.Dense(512, activation=layers.LeakyReLU(0.2))(x)
    x = layers.Dropout(0.4)(x)
    
    x = layers.Dense(256, activation=layers.LeakyReLU(0.2))(x)
    x = layers.Dropout(0.4)(x)
    
    output = layers.Dense(1, activation='sigmoid', name='validity')(x)
    
    model = Model(inputs=[sequence_input, label_input], outputs=output, name='Discriminator')
    
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.0002, beta_1=0.5),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    
    return model

## 4. GAN Model Function

In [21]:
def GAN_Model(generator, discriminator, latent_dim=100):
    discriminator.trainable = False
    
    noise_input = layers.Input(shape=(latent_dim,), name='gan_noise_input')
    label_input = layers.Input(shape=(1,), name='gan_label_input')
    
    generated_sequence = generator([noise_input, label_input])
    validity = discriminator([generated_sequence, label_input])
    
    gan = Model(inputs=[noise_input, label_input], outputs=validity, name='GAN')
    
    gan.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.0002, beta_1=0.5),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    
    return gan

## 5. Training Functions with Label Smoothing

In [22]:
def ConvertSequencesToOneHot(sequences, vocab_size):
    batch_size, seq_length = sequences.shape
    one_hot = np.zeros((batch_size, seq_length, vocab_size))
    
    for i in range(batch_size):
        for j in range(seq_length):
            if sequences[i, j] > 0 and sequences[i, j] < vocab_size:
                one_hot[i, j, sequences[i, j]] = 1
    
    return one_hot


def TrainGan(generator, discriminator, gan, X_train, y_train, 
             latent_dim=100, epochs=500, batch_size=64, sample_interval=50):
    vocab_size = generator.output_shape[-1]
    
    history = {
        'g_loss': [],
        'd_loss': [],
        'd_acc': [],
        'g_acc': [],
        'epoch': []
    }
    
    valid_smooth = np.ones((batch_size, 1)) * 0.9
    fake_smooth = np.zeros((batch_size, 1)) + 0.1
    
    for epoch in range(epochs):
        for _ in range(2):
            idx = np.random.randint(0, X_train.shape[0], batch_size)
            real_sequences = X_train[idx]
            real_labels = y_train[idx]
            
            real_sequences_onehot = ConvertSequencesToOneHot(real_sequences, vocab_size)
            
            noise = np.random.normal(0, 1, (batch_size, latent_dim))
            fake_labels = np.random.randint(0, np.max(y_train) + 1, (batch_size, 1))
            fake_sequences = generator.predict([noise, fake_labels], verbose=0)
            
            d_loss_real = discriminator.train_on_batch([real_sequences_onehot, real_labels], valid_smooth)
            d_loss_fake = discriminator.train_on_batch([fake_sequences, fake_labels], fake_smooth)
            d_loss = 0.5 * np.add(d_loss_real, d_loss_fake)
        
        noise = np.random.normal(0, 1, (batch_size, latent_dim))
        gen_labels = np.random.randint(0, np.max(y_train) + 1, (batch_size, 1))
        
        valid_targets = np.ones((batch_size, 1))
        g_loss = gan.train_on_batch([noise, gen_labels], valid_targets)
        
        if epoch % sample_interval == 0:
            history['epoch'].append(epoch)
            history['g_loss'].append(g_loss[0])
            history['d_loss'].append(d_loss[0])
            history['d_acc'].append(100 * d_loss[1])
            history['g_acc'].append(100 * g_loss[1])
            
            print(f"Epoch {epoch}/{epochs} - D Loss: {d_loss[0]:.4f}, D Acc: {100*d_loss[1]:.2f}%, G Loss: {g_loss[0]:.4f}, G Acc: {100*g_loss[1]:.2f}%")
    
    return history


def GenerateTextSamples(generator, tokenizer, label_encoder, num_samples=5, 
                        cefr_level='B2', latent_dim=100):
    label_idx = label_encoder[cefr_level]
    noise = np.random.normal(0, 1, (num_samples, latent_dim))
    labels = np.full((num_samples, 1), label_idx)
    
    generated_sequences = generator.predict([noise, labels], verbose=0)
    generated_indices = np.argmax(generated_sequences, axis=-1)
    
    reverse_word_index = {idx: word for word, idx in tokenizer.word_index.items()}
    
    generated_texts = []
    for sequence in generated_indices:
        words = [reverse_word_index.get(idx, '') for idx in sequence if idx > 0]
        text = ' '.join(words)
        generated_texts.append(text)
    
    return generated_texts

## 6. Configuration

In [23]:
LATENT_DIM = 100
MAX_SEQUENCE_LENGTH = 100
MAX_WORDS = 10000
EMBEDDING_DIM = 256
EPOCHS = 500
BATCH_SIZE = 64

TEXTS_PATH = 'cefr_leveled_texts.csv'
VOCAB_PATH = 'cefrj-vocabulary-profile-1.5.csv'
GRAMMAR_PATH = 'cefrj-grammar-profile-20180315.csv'

In [24]:
texts_df, vocab_df, grammar_df = LoadCefrData(TEXTS_PATH, VOCAB_PATH, GRAMMAR_PATH)
vocab_dict = CreateVocabularyDict(vocab_df)

X, y, tokenizer, label_encoder = PrepareSequences(
    texts_df, 
    max_sequence_length=MAX_SEQUENCE_LENGTH,
    max_words=MAX_WORDS
)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"\nTraining samples: {len(X_train)}")
print(f"Test samples: {len(X_test)}")

Loaded 1494 texts
CEFR Levels: ['B2' 'A2' 'C1' 'B1' 'A1' 'C2']

Loaded 7799 vocabulary entries
Loaded 500 grammar patterns
A1: 1064 words
A2: 2307 words
B1: 4446 words
B2: 6863 words
C1: 6863 words
C2: 6863 words

Sequence shape: (1494, 100)
Labels shape: (1494,)
Vocabulary size: 31289
Label encoding: {'A1': 0, 'A2': 1, 'B1': 2, 'B2': 3, 'C1': 4, 'C2': 5}

Training samples: 1195
Test samples: 299


In [25]:
print("Building Generator...")
generator = Generator(
    latent_dim=LATENT_DIM,
    vocab_size=MAX_WORDS,
    sequence_length=MAX_SEQUENCE_LENGTH,
    num_classes=len(label_encoder),
    embedding_dim=EMBEDDING_DIM
)
generator.summary()

Building Generator...


Model: "Generator"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ label_input         │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_2         │ (None, 1, 100)    │        600 │ label_input[0][0] │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ noise_input         │ (None, 100)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_2 (Flatten) │ (None, 100)       │          0 │ embedding_2[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_2       │ (None, 200)       │          0 │ noise_input[0][0… │
│ (Concatenate)       │                   │            │ flatten_2[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_7 (Dense)     │ (None, 512)       │    102,912 │ concatenate_2[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 512)       │      2,048 │ dense_7[0][0]     │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_4 (Dropout) │ (None, 512)       │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_8 (Dense)     │ (None, 1024)      │    525,312 │ dropout_4[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 1024)      │      4,096 │ dense_8[0][0]     │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_5 (Dropout) │ (None, 1024)      │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_9 (Dense)     │ (None, 2048)      │  2,099,200 │ dropout_5[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 2048)      │      8,192 │ dense_9[0][0]     │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_6 (Dropout) │ (None, 2048)      │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_10 (Dense)    │ (None, 25600)     │ 52,454,400 │ dropout_6[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape_2 (Reshape) │ (None, 100, 256)  │          0 │ dense_10[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_4 (LSTM)       │ (None, 100, 512)  │  1,574,912 │ reshape_2[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 100, 512)  │      2,048 │ lstm_4[0][0]      │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_5 (LSTM)       │ (None, 100, 512)  │  2,099,200 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 100, 512)  │      2,048 │ lstm_5[0][0]      │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼─────────────────

 Total params: 62,232,424 (237.40 MB)

 Trainable params: 62,223,208 (237.36 MB)

 Non-trainable params: 9,216 (36.00 KB)

In [26]:
print("\nBuilding Discriminator...")
discriminator = Discriminator(
    vocab_size=MAX_WORDS,
    sequence_length=MAX_SEQUENCE_LENGTH,
    num_classes=len(label_encoder),
    embedding_dim=EMBEDDING_DIM
)
discriminator.summary()


Building Discriminator...


Model: "Discriminator"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ label_input         │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_3         │ (None, 1, 100)    │        600 │ label_input[0][0] │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ sequence_input      │ (None, 100,       │          0 │ -                 │
│ (InputLayer)        │ 10000)            │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_3 (Flatten) │ (None, 100)       │          0 │ embedding_3[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ time_distributed_3  │ (None, 100, 256)  │  2,560,256 │ sequence_input[0… │
│ (TimeDistributed)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape_3 (Reshape) │ (None, 100, 1)    │          0 │ flatten_3[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_3       │ (None, 100, 257)  │          0 │ time_distributed… │
│ (Concatenate)       │                   │            │ reshape_3[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional_2     │ (None, 100, 1024) │  3,153,920 │ concatenate_3[0]… │
│ (Bidirectional)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_7 (Dropout) │ (None, 100, 1024) │          0 │ bidirectional_2[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional_3     │ (None, 100, 512)  │  2,623,488 │ dropout_7[0][0]   │
│ (Bidirectional)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_8 (Dropout) │ (None, 100, 512)  │          0 │ bidirectional_3[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional_4     │ (None, 256)       │    656,384 │ dropout_8[0][0]   │
│ (Bidirectional)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_13 (Dense)    │ (None, 512)       │    131,584 │ bidirectional_4[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_9 (Dropout) │ (None, 512)       │          0 │ dense_13[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_14 (Dense)    │ (None, 256)       │    131,328 │ dropout_9[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_10          │ (None, 256)       │          0 │ dense_14[0][0]    │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ validity (Dense)    │ (None, 1)         │        257 │ dropout_10[0][0]  │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 9,257,817 (35.32 MB)

 Trainable params: 9,257,817 (35.32 MB)

 Non-trainable params: 0 (0.00 B)

In [27]:
print("\nBuilding GAN...")
gan = GAN_Model(generator, discriminator, latent_dim=LATENT_DIM)
gan.summary()


Building GAN...


Model: "GAN"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ gan_noise_input     │ (None, 100)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ gan_label_input     │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Generator           │ (None, 100,       │ 62,232,424 │ gan_noise_input[… │
│ (Functional)        │ 10000)            │            │ gan_label_input[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Discriminator       │ (None, 1)         │  9,257,817 │ Generator[0][0],  │
│ (Functional)        │                   │            │ gan_label_input[… │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 71,490,241 (272.71 MB)

 Trainable params: 62,223,208 (237.36 MB)

 Non-trainable params: 9,267,033 (35.35 MB)

In [ ]:
print("\nTraining GAN...")
history = TrainGan(
    generator, 
    discriminator, 
    gan,
    X_train, 
    y_train,
    latent_dim=LATENT_DIM,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    sample_interval=50
)


Training GAN...
Epoch 0/500 - D Loss: 0.6929, D Acc: 0.00%, G Loss: 0.6930, G Acc: 79.69%


In [ ]:
print("\nGenerating sample texts at different CEFR levels...")

for level in ['A1', 'A2', 'B1', 'B2']:
    if level in label_encoder:
        print(f"\n{'='*60}")
        print(f"CEFR Level: {level}")
        print('='*60)
        
        samples = GenerateTextSamples(
            generator, 
            tokenizer, 
            label_encoder,
            num_samples=3,
            cefr_level=level,
            latent_dim=LATENT_DIM
        )
        
        for i, text in enumerate(samples, 1):
            print(f"\nSample {i}:")
            print(text)

In [ ]:
generator.save('cefr_text_generator.keras')
discriminator.save('cefr_text_discriminator.keras')
print("\nModels saved successfully!")

## 7. Utility Functions for Enhanced Text Generation

In [ ]:
def FilterTextByCefrVocabulary(text, cefr_level, vocab_dict):
    allowed_words = vocab_dict[cefr_level]
    words = text.split()
    
    filtered_words = []
    invalid_count = 0
    
    for word in words:
        clean_word = re.sub(r'[^a-z]', '', word.lower())
        if clean_word in allowed_words or clean_word == '':
            filtered_words.append(word)
        else:
            invalid_count += 1
    
    filtered_text = ' '.join(filtered_words)
    is_valid = invalid_count / max(len(words), 1) < 0.15
    
    return filtered_text, is_valid


def GenerateHighQualitySamples(generator, tokenizer, label_encoder, vocab_dict,
                               cefr_level='B2', num_samples=10, latent_dim=100):
    valid_texts = []
    attempts = 0
    max_attempts = num_samples * 5
    
    while len(valid_texts) < num_samples and attempts < max_attempts:
        candidates = GenerateTextSamples(
            generator, 
            tokenizer, 
            label_encoder,
            num_samples=5,
            cefr_level=cefr_level,
            latent_dim=latent_dim
        )
        
        for text in candidates:
            filtered_text, is_valid = FilterTextByCefrVocabulary(
                text, cefr_level, vocab_dict
            )
            
            if is_valid and len(filtered_text.split()) > 5:
                valid_texts.append(filtered_text)
                
                if len(valid_texts) >= num_samples:
                    break
        
        attempts += 1
    
    return valid_texts[:num_samples]


def PlotTrainingHistory(history):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
    
    ax1.plot(history['epoch'], history['d_loss'], label='Discriminator Loss', linewidth=2)
    ax1.plot(history['epoch'], history['g_loss'], label='Generator Loss', linewidth=2)
    ax1.set_xlabel('Epoch', fontsize=12)
    ax1.set_ylabel('Loss', fontsize=12)
    ax1.set_title('GAN Training Loss', fontsize=14, fontweight='bold')
    ax1.legend(fontsize=10)
    ax1.grid(True, alpha=0.3)
    
    ax2.plot(history['epoch'], history['d_acc'], label='Discriminator Accuracy', linewidth=2, color='green')
    if 'g_acc' in history:
        ax2.plot(history['epoch'], history['g_acc'], label='Generator Accuracy', linewidth=2, color='orange')
    ax2.set_xlabel('Epoch', fontsize=12)
    ax2.set_ylabel('Accuracy (%)', fontsize=12)
    ax2.set_title('GAN Training Accuracy', fontsize=14, fontweight='bold')
    ax2.legend(fontsize=10)
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('gan_training_history.png', dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:
print("Generating high-quality samples with vocabulary filtering...")

hq_samples = GenerateHighQualitySamples(
    generator,
    tokenizer,
    label_encoder,
    vocab_dict,
    cefr_level='B2',
    num_samples=5,
    latent_dim=LATENT_DIM
)

print(f"\n{'='*60}")
print(f"High-Quality B2 Level Samples")
print('='*60)
for i, text in enumerate(hq_samples, 1):
    print(f"\n{i}. {text}")

PlotTrainingHistory(history)